# ML 03 – Bank Customer Churn (ANN)

Run the cells from top to bottom. Update dataset paths or column names if your CSV uses different names.

## Dataset
Place `Churn_Modelling.csv` in `../datasets/`. Expected target column: `Exited`. Typical columns include `CreditScore`, `Geography`, `Gender`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, and `EstimatedSalary`. Identifier/name columns are excluded.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neural_network import MLPClassifier

DATA_PATH = "../datasets/Churn_Modelling.csv"
df = pd.read_csv(DATA_PATH)
print(df.head())
print(df.columns.tolist())

In [ ]:
TARGET = "Exited"
if TARGET not in df.columns:
    raise ValueError(f"Target '{TARGET}' not found. Update TARGET to match your CSV.")
drop_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
X = df.drop(columns=[TARGET] + drop_cols)
y = df[TARGET]

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = Pipeline([
    ("preprocess", preprocess),
    ("ann", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                          solver="adam", max_iter=300, random_state=42,
                          early_stopping=True))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

## Notes
This uses scikit-learn's multilayer perceptron (ANN). It is a practical baseline; if your lab specifically requires TensorFlow/Keras, adapt the model section accordingly.